# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset name and description
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets, and for each, list its fields and columns with their `@id`s.

In [ ]:
# Explore dataset metadata for record sets, using only @id for referencing

def print_record_sets(ds):
    rs_list = ds.metadata.record_sets
    print(f"\nRecord Sets ({len(rs_list)}):\n{'-'*40}")
    for rs in rs_list:
        print(f"RecordSet @id: {rs.id}, name: {rs.name if hasattr(rs,'name') else ''}")
        if hasattr(rs, 'fields'):
            print("  Fields:")
            for field in rs.fields:
                print(f"    Field @id: {field.id}, name: {getattr(field, 'name', '')}, dataType: {getattr(field, 'data_type', None)}")
        if hasattr(rs, 'columns'):
            print("  Columns:")
            for col in rs.columns:
                print(f"    Column @id: {col.id}, name: {getattr(col, 'name', '')}")
        print()

# Show the record sets structure
print_record_sets(dataset)

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All referencing is done via `@id` fields.

In [ ]:
# Extract data for all record sets
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

# List sample records for each set
for record_set_id in record_set_ids:
    print(f"\nLoading records from set: {record_set_id}")
    # Extract records as dicts
    records = list(dataset.records(record_set=record_set_id))
    # If records present, convert to DataFrame
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"  Columns: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head())
    else:
        print("  No records found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on numerical abundance, normalizing a numeric field, and grouping by a category. All references use `@id` fields from the earlier step.

In [ ]:
# Select the primary record set for analysis (typically proteomics or main table)
# You may need to update this to match the main data set @id. We'll infer the first set.
if len(dataframes) == 0:
    print("No tabular dataframes to analyze.")
else:
    main_record_set_id = list(dataframes.keys())[0]
    df = dataframes[main_record_set_id]

    # Heuristically choose a numeric field/column by looking for a likely numerical column
    print(f"Sample columns in {main_record_set_id}: {df.columns.tolist()}")
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_candidates) == 0:
        print("No numeric fields found! Showing the first few records for manual inspection.")
        display(df.head())
    else:
        # Pick the most likely numeric field (e.g., 'abundance', 'MW', etc. if present)
        print(f"Numeric fields: {numeric_candidates}")
        numeric_field = numeric_candidates[0]
        print(f"Analyzing numeric field: '{numeric_field}' (use @id from column header)")
        # Set a threshold for filtering, e.g., >10
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold} (count: {len(filtered_df)}):")
        display(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Find a likely categorical/group field (not numeric, not id field)
        possible_group_fields = [c for c in df.columns if df[c].dtype == 'object' and not c.lower().endswith('id')]
        if len(possible_group_fields) > 0:
            group_field = possible_group_fields[0]
            print(f"Grouping by '{group_field}' (as possible category)")
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            display(grouped_df.head())
        else:
            print("No suitable group/categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of a numeric field and the group means if a group field was detected.

In [ ]:
if len(dataframes) == 0:
    print('No data available for visualization.')
elif len(numeric_candidates) > 0:
    # Histogram of numeric field
    plt.figure(figsize=(7,4))
    df[numeric_field].hist(bins=40)
    plt.title(f"Distribution of {numeric_field} in {main_record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If grouping was done, plot top group means
    if 'grouped_df' in locals():
        grouped_df[numeric_field].sort_values(ascending=False).head(10).plot(kind='barh')
        plt.title(f"Top 10 {group_field} means of {numeric_field}")
        plt.xlabel(f"Mean of {numeric_field}")
        plt.ylabel(group_field)
        plt.show()

## 6. Conclusion

This notebook demonstrated how to access and explore the FAIR^2 dataset ([Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)) programmatically using the `mlcroissant` library.

- The record sets, fields, and columns were referenced and listed by their `@id` field as required by the Croissant schema standard.
- Tabular data from each record set was loaded into a DataFrame for exploratory analysis.
- Simple filtering, normalization, and grouping operations were shown using numeric and categorical fields (where available).
- Visualizations to explore distributions and group means were included.

For your own analysis:
- Adjust which record set and fields are analyzed based on your scientific question.
- Use the `@id` fields when specifying data extraction via `mlcroissant`.

See [mlcroissant documentation](https://mlcommons.org/croissant/) for further details and advanced workflow options.